# Kapitel 12: Training Ihres Modells

> "Ich bin nicht gescheitert. Ich habe nur 10.000 Wege gefunden, die nicht funktionieren."
> — **Thomas Edison**, Erfinder

---

## Was Sie lernen werden

- Wie Next-Token-Vorhersage Modelle das Schreiben lehrt
- Die 5-Schritte-Trainingsrezept, das alle neuronalen Netze antreibt
- Warum Verlustkurven zeigen, ob Ihr Modell lernt
- Wie Sie Überanpassung verhindern und wissen, wann Sie stoppen sollten
- Speichern von Checkpoints, damit Sie niemals Fortschritt verlieren
- Der Nervenkitzel, Ihr Modell von Kauderwelsch zu Kohärenz verwandeln zu sehen

---

## Setup

Zuerst installieren wir die benötigten Pakete:

In [ ]:
# Benötigte Pakete installieren
!pip install -q torch transformers tqdm

In [ ]:
# ===== IMPORTS =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Gerät festlegen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwendetes Gerät: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== REPRODUZIERBARKEIT =====
def set_seed(seed=42):
    """Setzt alle Seeds für Reproduzierbarkeit."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## 1. Modellkomponenten aus Kapiteln 10-11

Zuerst holen wir das MiniGPT-Modell, das wir in den vorherigen Kapiteln erstellt haben.

In [ ]:
# ===== MULTI-HEAD ATTENTION (aus Kapitel 10) =====

class MultiHeadAttention(nn.Module):
    """Effiziente mehrköpfige Aufmerksamkeit (bündelt alle Köpfe zusammen)."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = attn_weights @ V
        attn_output = attn_output.transpose(1, 2).reshape(batch, seq, d_model)

        return self.out_proj(attn_output), attn_weights

print("MultiHeadAttention definiert!")

In [ ]:
# ===== FEEDFORWARD-NETZWERK (aus Kapitel 10) =====

class FeedForward(nn.Module):
    """Positionsweises Feedforward-Netzwerk."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print("FeedForward definiert!")

In [ ]:
# ===== TRANSFORMER-BLOCK (aus Kapitel 10) =====

class TransformerBlock(nn.Module):
    """Vollständiger Transformer-Block (Prä-Normalisierung wie GPT-2)."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, attn_weights

print("TransformerBlock definiert!")

In [ ]:
# ===== GPT-KONFIGURATION (aus Kapitel 11) =====

@dataclass
class GPTConfig:
    """Konfiguration für das MiniGPT-Modell."""
    vocab_size: int = 50257
    max_seq_len: int = 1024
    embed_dim: int = 768
    num_heads: int = 12
    num_layers: int = 12
    d_ff: int = 3072
    dropout: float = 0.1

    def __post_init__(self):
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) muss durch num_heads ({self.num_heads}) teilbar sein"

print("GPTConfig definiert!")

In [ ]:
# ===== MINIGPT-MODELL (aus Kapitel 11) =====

class MiniGPT(nn.Module):
    """Ein minimales GPT-Sprachmodell."""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # Einbettungen
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer-Blöcke
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # Finale Schichtnormalisierung und LM-Kopf
        self.ln_f = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # Gewichtsverknüpfung
        self.lm_head.weight = self.token_embed.weight

        # Gewichte initialisieren
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        batch, seq = token_ids.shape
        device = token_ids.device

        tok_emb = self.token_embed(token_ids)
        positions = torch.arange(seq, device=device)
        pos_emb = self.pos_embed(positions)
        x = self.dropout(tok_emb + pos_emb)

        mask = torch.tril(torch.ones(seq, seq, device=device))

        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if return_attention:
            return logits, attention_weights
        return logits

print("MiniGPT-Klasse definiert!")

## 2. Datensatz herunterladen

Wir verwenden TinyShakespeare - klein genug, um in Minuten zu trainieren.

In [ ]:
# TinyShakespeare herunterladen
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Datensatzgröße: {len(text):,} Zeichen")
print(f"\nBeispiel:\n{text[:500]}")

## 3. Dataset und DataLoader erstellen

In [ ]:
class TextDataset(Dataset):
    """Einfacher Datensatz, der Textblöcke zurückgibt."""

    def __init__(self, text, chunk_size=256):
        self.chunks = []
        for i in range(0, len(text) - chunk_size, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


def collate_fn(batch, tokenizer, max_length=128):
    """Tokenisiert und fügt Padding zu einem Batch von Textstrings hinzu."""
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Datensatz erstellen
dataset = TextDataset(text, chunk_size=256)
print(f"Anzahl der Chunks: {len(dataset)}")

In [ ]:
# Train/Val-Aufteilung erstellen
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Training: {len(train_dataset)}, Validierung: {len(val_dataset)}")

# DataLoaders erstellen
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Einen Batch überprüfen
input_ids, attention_mask = next(iter(train_loader))
print(f"Batch input_ids Form: {input_ids.shape}")

## 4. Modell erstellen

In [ ]:
# Kleine Konfiguration für schnelles Training
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

## 5. Der "Vorher"-Zustand: Untrainiertes Modell

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=30, temperature=1.0):
    """Generiert Text mit Temperaturkontrolle."""
    model.eval()
    device = next(model.parameters()).device

    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        logits = model(token_ids)
        next_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        token_ids = torch.cat([token_ids, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(token_ids[0])


# Aus untrainiertem Modell generieren
print("VOR DEM TRAINING (zufällige Gewichte):")
print("="*50)
prompt = "The king"
print(f"Prompt: '{prompt}'")
print(f"Ausgabe: {generate(model, tokenizer, prompt)}")
print("\n(Zufälliges Kauderwelsch - das Modell hat noch nichts gelernt!)")

## 6. Training-Setup

In [ ]:
# Training-Hyperparameter
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Gesamte Trainingsschritte: {total_steps}")

# Optimierer und Scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Training- und Evaluierungsfunktionen

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """Trainiert eine Epoche mit dem 5-Schritte-Rezept."""
    model.train()
    total_loss = 0

    progress = tqdm(dataloader, desc="Training")
    for input_ids, attention_mask in progress:
        input_ids = input_ids.to(device)

        # Verschieben für Sprachmodellierung
        inputs = input_ids[:, :-1]
        targets = input_ids[:, 1:]

        # ===== DAS 5-SCHRITTE-REZEPT =====
        optimizer.zero_grad(set_to_none=True)       # 1. Gradienten nullsetzen
        logits = model(inputs)                       # 2. Vorwärtsdurchlauf
        loss = F.cross_entropy(                      # 3. Verlust berechnen
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape behandelt nicht-zusammenhängende Slices
            ignore_index=tokenizer.pad_token_id
        )
        loss.backward()                              # 4. Rückwärtsdurchlauf
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()                             # 5. Gewichte aktualisieren
        scheduler.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Evaluiert das Modell und berechnet die Perplexität."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        inputs = input_ids[:, :-1]
        targets = input_ids[:, 1:]

        logits = model(inputs)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape behandelt nicht-zusammenhängende Slices
            ignore_index=tokenizer.pad_token_id,
            reduction='sum'
        )

        mask = (targets != tokenizer.pad_token_id)
        total_loss += loss.item()
        total_tokens += mask.sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """Speichert einen Training-Checkpoint."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }, path)
    print(f"Checkpoint gespeichert: {path}")

## 8. Trainingsschleife

In [ ]:
# Training!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoche {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Trainieren
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Evaluieren
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nTraining-Verlust: {train_loss:.4f}")
    print(f"Val-Verlust:      {val_loss:.4f}")
    print(f"Perplexität:      {perplexity:.1f}")

    # Besten Checkpoint speichern
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nTraining abgeschlossen!")
print(f"Bester Validierungsverlust: {best_val_loss:.4f}")

## 9. Verlustkurven zeichnen

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Training-Verlust')
plt.plot(epochs, val_losses, 'r-o', label='Validierungs-Verlust')
plt.xlabel('Epoche')
plt.ylabel('Verlust')
plt.title('Training- und Validierungsverlust')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Die Belohnung: Textgenerierung!

In [ ]:
# Bestes Modell laden
checkpoint = torch.load("best_model.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Checkpoint geladen aus Epoche {checkpoint['epoch'] + 1}")
print(f"Validierungsverlust: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("NACH DEM TRAINING (auf Shakespeare):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nPrompt: '{prompt}'")
    print(f"Ausgabe: {output}")
    print("-"*40)

## 11. Vorher-Nachher-Vergleich

In [ ]:
# Neues untrainiertes Modell zum Vergleich erstellen
set_seed(123)  # Anderer Seed für unterschiedliche zufällige Gewichte
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"Prompt: '{prompt}'")
print("="*60)
print("\nVORHER (untrainiert):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nNACHHER (auf Shakespeare trainiert):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nGleiche Architektur. Gleicher Code. Training macht den ganzen Unterschied!")

## 12. Temperaturvergleich

In [ ]:
prompt = "The noble lord"

print(f"Prompt: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Temperatur {temp}:")
    print(f"  {output}")
    print()

## Zusammenfassung

**Was wir gebaut haben:**

1. **Label-Verschiebung** für Next-Token-Vorhersage
2. **DataLoaders**, die Text effizient batchen und tokenisieren
3. **Das 5-Schritte-Trainingsrezept**: zero_grad → forward → loss → backward → step
4. **Evaluierung** mit Validierungsverlust und Perplexität
5. **Checkpointing** zum Speichern und Fortsetzen des Trainings
6. **Textgenerierung** mit Temperaturkontrolle

**Schlüsselkonzepte:**

- Training ist eine Feedback-Kontrollschleife: Fehler messen, Gewichte anpassen, wiederholen
- Kreuzentropie-Verlust bestraft zuversichtliche falsche Vorhersagen stärker als unsichere
- Überanpassung = Auswendiglernen von Trainingsdaten (Training-Verlust ↓, Val-Verlust ↑)
- Perplexität misst, wie "überrascht" das Modell ist - niedriger ist besser
- Temperatur kontrolliert die Generierungsvielfalt

**Weiter:** Kapitel 13 wird Ihnen beibringen, dieses Modell für spezifische Aufgaben fein abzustimmen!

## Übungen

### Übung 1: Lernraten-Experiment

Probieren Sie verschiedene Lernraten aus und vergleichen Sie die Ergebnisse.

In [ ]:
# IHR CODE HIER
# Trainieren Sie mit lr=1e-5, lr=3e-4, lr=1e-2
# Vergleichen Sie die Verlustkurven
# Welche Lernrate funktioniert am besten?

### Übung 2: Mehr Epochen

In [ ]:
# IHR CODE HIER
# Trainieren Sie für 5-10 Epochen statt 3
# Verbessert sich das Modell weiter?
# Sehen Sie Anzeichen von Überanpassung?

### Übung 3: Modellgrößen-Vergleich

In [ ]:
# IHR CODE HIER
# Erstellen Sie ein kleineres Modell (2 Schichten, 128 Dimensionen)
# Erstellen Sie ein größeres Modell (6 Schichten, 384 Dimensionen)
# Vergleichen Sie Trainingsgeschwindigkeit und finale Perplexität